# Linear-density turning point and ray sheets

This notebook reruns the bundled 20-degree single-ray example. For $n_e/n_{\rm crit}=x/L$ and vacuum incidence at angle $\alpha$ from the density-gradient direction, conservation of transverse momentum gives $n_{e,\rm turn}/n_{\rm crit}=\cos^2\alpha$.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyGATH.io import load_simulation_config
from pyGATH.plotting import plot_hydro_slice, plot_ray_trajectories
from pyGATH.raytracing import RAY_STATE_LAYOUT, critical_density

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "configs").is_dir():
    repo_root = repo_root.parent

deck = repo_root / "configs" / "test_configs" / "linear_gradient_turning.toml"
print(f"Loading simulation config from {deck}...")
time_start = time.time()
simulation = load_simulation_config(deck)
time_end = time.time()
print(f"Loaded simulation config: {simulation}")
print(f"    Simulation config loaded in {time_end - time_start:.2e} seconds")
print()

print("Building grid and initializing rays...")
time_start = time.time()
grid = simulation.build_grid()
time_end = time.time()
print(f"Grid loaded. Grid shape: {grid.ncells}")
print(f"    Grid loaded in {time_end - time_start:.2e} seconds")
print()

print("Loading beams...")
time_start = time.time()
beams = simulation.load_beams()
time_end = time.time()
print(f"Loaded beams: {beams.names}")
print(f"    Beams loaded in {time_end - time_start:.2e} seconds")
print()

print("Initializing rays...")
time_start = time.time()
initial_rays = simulation.initialize_rays(grid, beams=beams)
time_end = time.time()
print("Ray initialisation complete:")
print(f"    Ray initialisation time: {time_end - time_start:.2e} seconds")
print(f"    Number of beams: {initial_rays.state.shape[0]}")
print(f"    Number of rays axis 1: {initial_rays.state.shape[1]}")
print(f"    Number of rays axis 2: {initial_rays.state.shape[2]}")
print(f"    Number of ray state variables: {initial_rays.state.shape[3]}")
print()

print("Tracing rays...")
time_start = time.time()
result = simulation.trace_rays(initial_rays, grid)
time_end = time.time()
print("Tracing complete:")
print(f"    Tracing time: {time_end - time_start:.2e} seconds")
print(f"    Number of beams: {result.sheet_fields.shape[0]}")
print(f"    Number of sheets: {result.sheet_fields.shape[1]}")
print(f"    Number of rays axis 1: {result.sheet_fields.shape[2]}")
print(f"    Number of rays axis 2: {result.sheet_fields.shape[3]}")
print(f"    Number samples per sheet: {result.sheet_fields.shape[4]}")
print(f"    Number of ray state variables: {result.sheet_fields.shape[5]}")
print()

## Check the analytical turn and sheet split

In [ ]:
direction = np.asarray(beams.direction[0])
angle = np.arctan2(direction[1], direction[0])
ncritical = float(critical_density(beams.omega[0]))
density_gradient = float(grid.grad_ne[0, 0, 0, 0])
density_scale = ncritical / density_gradient
expected_turn_x = density_scale * np.cos(angle) ** 2

fields = np.asarray(result.sheet_fields[0, :, 0, 0])
numerical_turn_x = np.max(fields[..., RAY_STATE_LAYOUT.position.start])
caustic_x = fields[0, -1, RAY_STATE_LAYOUT.position.start]

assert bool(result.has_caustic[0, 0, 0])
assert result.sheet_fields.shape[1] == 2
np.testing.assert_allclose(numerical_turn_x, expected_turn_x, rtol=2e-3)
np.testing.assert_allclose(caustic_x, expected_turn_x, rtol=3e-3)
print(f"Analytical turn: {expected_turn_x:.8e} m")
print(f"Numerical turn:  {numerical_turn_x:.8e} m")
print(f"Tube caustic:   {caustic_x:.8e} m")

## Electron density and ray trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
_, density_mesh = plot_hydro_slice(
    grid,
    "ne",
    axes=("x", "y"),
    index=0,
    projection="xy",
    normalization=ncritical,
    ax=ax,
    cmap="viridis",
    vmin=0.0,
    vmax=1.1,
)
plot_ray_trajectories(result, beam_index=0, projection="xy", ax=ax)
fig.colorbar(density_mesh, ax=ax, label=r"$n_e/n_{\rm crit}$")
ax.legend()
ax.set_aspect("equal")
ax.set_title("20-degree ray in a linear density gradient")
fig.tight_layout()

In [ ]:
plt.scatter(
    result.sheet_fields[0, 0, 0, 0, :, RAY_STATE_LAYOUT.position.start],
    result.sheet_fields[0, 0, 0, 0, :, RAY_STATE_LAYOUT.position.start + 1],
    color="red",
    marker="x",
    s=100,
    label="Ray caustic",
)
plt.scatter(
    result.sheet_fields[0, 1, 0, 0, :, RAY_STATE_LAYOUT.position.start],
    result.sheet_fields[0, 1, 0, 0, :, RAY_STATE_LAYOUT.position.start + 1],
    color="blue",
    marker="x",
    s=100,
    label="Ray caustic",
)